# Uso De Transformrs Para Criaçao De Resumos De Videos Do **Youtube**

---


# Introdução
Este notebook Colab é projetado para utilizar modelos de Transformers para criar resumos de vídeos do YouTube. Ele abrange desde a extração de áudio dos vídeos até a geração de resumos utilizando modelos de linguagem avançados. A seguir, uma descrição detalhada de cada célula do notebook.


---


OBS: Apos teste foi indetificado que o modelo nao suporta Videos acima 20 minutos


# Instalaçao de Bibliotecas

In [ ]:
# Instalar as bibliotecas necessárias
!pip install yt-dlp pydub openai-whisper # Instala yt-dlp para download de vídeos, pydub para manipulação de áudio e openai-whisper para transcrição de áudio.
!apt-get install ffmpeg # Instala o ffmpeg para manipulação de mídia.
!pip install transformers # Instala a biblioteca transformers para trabalhar com modelos de linguagem.
!pip install transformers datasets # Instala transformers e datasets para uso com datasets.
!pip install transformers accelerate -q # Instala transformers e accelerate para treinamento acelerado.

!pip install transformers # Instala transformers novamente (redundante, pode ser removido).
!pip install einops accelerate bitsandbytes # Instala bibliotecas para otimização de modelos.
!pip install sentence_transformers # Instala sentence_transformers para codificação de frases.

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.


# **Verificação da GPU:** Comando (`!nvidia-smi`) é usado para verificar se uma GPU está disponível para acelerar o processamento.

In [ ]:
!nvidia-smi

Sun Nov 17 02:23:12 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   46C    P0              27W /  70W |    663MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

# Função para Baixar Vídeos do YouTube

---

Esta célula define uma função para baixar o áudio de vídeos do YouTube e salvá-lo como um arquivo MP3.

In [ ]:
#Importar as bibliotecas
import yt_dlp
from pydub import AudioSegment
import whisper
import os

# Função para baixar o vídeo do YouTube
def download_audio_from_youtube(video_url, output_filename='audio.mp3'):
    try:
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': output_filename,
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '1080',
            }],
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
        print(f"Áudio baixado com sucesso: {output_filename}")
    except Exception as e:
        print(f"Erro ao baixar o vídeo: {e}")

# Função para Converter Áudio para WAV

---

Esta célula define uma função para converter arquivos de áudio MP3 para o formato WAV.

In [ ]:
# Função para converter o áudio para WAV
def convert_audio_to_wav(input_filename, output_filename='audio.wav'):
    try:
        audio = AudioSegment.from_file(input_filename)
        audio.export(output_filename, format='wav')
        print(f"Áudio convertido com sucesso: {output_filename}")
    except Exception as e:
        print(f"Erro ao converter o áudio: {e}")

# Função para Transcrever Áudio para Texto

---

Esta célula define uma função para transcrever o áudio para texto usando o modelo Whisper.

---
O Whisper é um modelo de reconhecimento automático de fala (ASR) desenvolvido pela OpenAI. Ele é projetado para ser preciso e robusto em uma variedade de idiomas e sotaques.

**Funcionalidade:**

O Whisper é capaz de transcrever áudio em vários idiomas, incluindo português, e pode lidar com diferentes níveis de ruído e qualidade de áudio.

**Vantagens:**

* **Precisão:** O Whisper oferece uma alta taxa de precisão na transcrição de áudio, mesmo em situações desafiadoras.
* **Suporte a Múltiplos Idiomas:** Ele suporta uma variedade de idiomas, tornando-o útil para uma gama maior de usuários.
* **Robustez:** Ele é relativamente robusto em relação a ruídos e variações na qualidade do áudio.




In [ ]:
from random import sample
# Função para transcrever o áudio para texto
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")
processor = WhisperProcessor.from_pretrained("openai/whisper-base")

def transcribe_audio_to_text(audio_filename, output_text_filename='transcription.txt'):
    try:
        audio_data, sampling_rate = librosa.load(audio_filename, sr=16000)
        inputs = processor(audio_data, sampling_rate=sampling_rate, return_tensors="pt",language='en')
        transcription = inputs.input_features
        transcricao = model.generate(transcription)
        result = processor.decode(transcricao[0], skip_special_tokens=True)
        with open(output_text_filename, 'w') as f:
            f.write(result)
        print(f"Transcrição concluída e salva em '{output_text_filename}'")
    except Exception as e:
        print(f"Erro ao transcrever o áudio: {e}")

# Chamando funçoes

In [ ]:
# URL do vídeo do YouTube
video_url = input("Digite a URL do vídeo do YouTube: ")

# Passo 1: Baixar o vídeo do YouTube
download_audio_from_youtube(video_url, 'audio.mp3')

# Verificar se o arquivo baixado existe e ajustar o nome do arquivo, se necessário
if not os.path.exists('audio.mp3') and os.path.exists('audio.mp3.mp3'):
    os.rename('audio.mp3.mp3', 'audio.mp3')

# Passo 2: Converter o áudio para WAV
convert_audio_to_wav('audio.mp3', 'audio.wav')

# Passo 3: Transcrever o áudio para texto
transcribe_audio_to_text('audio.wav')

Digite a URL do vídeo do YouTube: https://www.youtube.com/watch?v=JxKEqrOeu1A
[youtube] Extracting URL: https://www.youtube.com/watch?v=JxKEqrOeu1A
[youtube] JxKEqrOeu1A: Downloading webpage
[youtube] JxKEqrOeu1A: Downloading ios player API JSON
[youtube] JxKEqrOeu1A: Downloading mweb player API JSON
[youtube] JxKEqrOeu1A: Downloading m3u8 information
[info] JxKEqrOeu1A: Downloading 1 format(s): 251
[download] Destination: audio.mp3
[download] 100% of   14.27MiB in 00:00:01 at 13.80MiB/s  
[ExtractAudio] Destination: audio.mp3.mp3
Deleting original file audio.mp3 (pass -k to keep)
Áudio baixado com sucesso: audio.mp3
Áudio convertido com sucesso: audio.wav
Transcrição concluída e salva em 'transcription.txt'


# Carregando o Modelo e T5

---
Este notebook demonstra o uso do modelo T5 (Text-To-Text Transfer Transformer) para gerar resumos de vídeos do YouTube. O T5 é um modelo de linguagem poderoso e versátil, desenvolvido pelo Google, que pode ser usado para uma variedade de tarefas de processamento de linguagem natural (PLN), incluindo tradução automática, resposta a perguntas, sumarização de texto e muito mais.

**Como o T5 funciona:**

O T5 é baseado na arquitetura Transformer, que permite que ele processe e gere texto com alta precisão. Ele é treinado em um grande conjunto de dados de texto, o que lhe permite aprender as relações complexas entre palavras e frases.

**Aplicações em Sumarização:**

O T5 é particularmente eficaz para a tarefa de sumarização de texto, pois ele pode aprender a identificar as informações mais importantes em um texto e gerar um resumo conciso e informativo. Neste notebook, utilizaremos o T5 para gerar resumos de transcrições de áudio de vídeos do YouTube, permitindo uma compreensão rápida do conteúdo do vídeo.

**Modelos Utilizados:**

O notebook utiliza um modelo T5 pré-treinado em português ("recogna-nlp/ptt5-base-summ-cstnews"), otimizado para a tarefa de sumarização de notícias, e um tokenizador ("unicamp-dl/ptt5-base-portuguese-vocab") compatível com a linguagem portuguesa.


In [ ]:
# Carregando modelo
from transformers import T5Model, T5ForConditionalGeneration , T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("unicamp-dl/ptt5-base-portuguese-vocab")
model = T5ForConditionalGeneration.from_pretrained("unicamp-dl/ptt5-base-portuguese-vocab")

# Carrega o arquivo de transcriçao do video

In [ ]:
with open("/content/transcription.txt", "r") as f:
      transcript_text = f.read()

transcript_text

' Olha, a proposta de menina da Constituição apresentada pela deputada Érica Rilton, que visa modificar as leis trabalhistas para reduzir a jornada de trabalho no Brasil, voltou a gerar fortes reações nos brasileiros no web, após ser um dos assuntos mais comentados na rede social. O presidente Lula está tirando o foco da situação e prefere aguardar que o tema toma corpo dentro do Congresso. Para ele, o assunto ainda precisa ser debatido para chegar a um consciência.'

# Definir o Texto de Entrada

---
Esta célula define o texto que será resumido pelo modelo.


In [ ]:
text = transcript_text

# Geração do Resumo

---
Esta célula usa o modelo T5 para gerar o resumo do texto tokenizado.


In [ ]:
inputs = tokenizer.encode(text, max_length=1024, truncation=True, return_tensors='pt')
summary_ids = model.generate(inputs, max_length=1000, min_length=32, num_beams=5, no_repeat_ngram_size=3, early_stopping=True)
summary = tokenizer.decode(summary_ids[0])

# Quebrar a linha a cada 50 caracteres
print(summary.replace('\n', '\n\n'))

<pad> Olha, a proposta de menina da Constituição apresentada pela deputada Érica Rilton, que visa modificar as leis trabalhistas para reduzir a jornada de trabalho no Brasil, voltou a gerar fortes reações nos brasileiros no web, após ser um dos assuntos mais comentados na rede social. O presidente Lula está tirando o foco da situação e prefere aguardar que o tema toma corpo dentro do Congresso. Para ele, o assunto ainda precisa ser debatido para chegar a um consciência.</s>


# Modelos Bart

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

bart_inputs = tokenizer.encode(text, max_length=1024, truncation=True, return_tensors='pt')
bart_outputs = model.generate(bart_inputs, max_length=1000, min_length=32, num_beams=5, no_repeat_ngram_size=3, early_stopping=True)
summary = tokenizer.decode(bart_outputs[0])

# Quebrar a linha a cada 50 caracteres
print(summary.replace('\n', '\n\n'))

</s><s>A proposta de menina da Constituição voltou a gerar fortes reações nos brasileiros no web. O presidente Lula está tirando o foco da situação.</s>
